I.) Finding the miRNA targets from a database and match them to the corresponding genes in the PFC expression data. 

1.) Import libraries


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
#import GEOparse


2.) Load  raw miRDB, that provides experimental evidence for miRNA targets

In [2]:
mirdb_raw = pd.read_csv(
    "data/miRDB_v6.0_prediction_result.txt.gz",
    sep="\t",
    header=None,
    names=["miRNA", "RefSeq", "miRDB_score"],
    dtype={
        "miRNA": "string",
        "RefSeq": "string",
        "miRDB_score": "float64"
    }
)

mirdb_raw.head()

,miRNA,RefSeq,miRDB_score
0,cfa-miR-1185,XM_537211,59.343810
1,cfa-miR-1185,XM_536047,54.527000
2,cfa-miR-1185,XM_005617022,55.171633
3,cfa-miR-1185,XM_014117861,57.440906
4,cfa-miR-1185,XM_014107884,57.151900


3.)Load human predictions and select filter cutoff of >= 85% miRDB_score

In [3]:
mirdb_human = mirdb_raw[
    mirdb_raw["miRNA"].str.startswith("hsa-", na=False)
].copy()

mirdb_high_conf = mirdb_human[
    mirdb_human["miRDB_score"] >= 80
].copy()

print("All predictions:", len(mirdb_raw))
print("Human predictions:", len(mirdb_human))
print("High-confidence human predictions:", len(mirdb_high_conf))
print("Unique human miRNAs:", mirdb_high_conf["miRNA"].nunique())
print("Unique RefSeq transcripts:", mirdb_high_conf["RefSeq"].nunique())

mirdb_high_conf.head()

All predictions: 6831595
Human predictions: 3375741
High-confidence human predictions: 812240
Unique human miRNAs: 2638
Unique RefSeq transcripts: 38380


,miRNA,RefSeq,miRDB_score
1776178,hsa-let-7a-2-3p,NM_001130845,89.899500
1776181,hsa-let-7a-2-3p,NM_014423,80.149571
1776186,hsa-let-7a-2-3p,NM_001207024,84.703100
1776192,hsa-let-7a-2-3p,NM_020529,94.817070
1776195,hsa-let-7a-2-3p,NM_002970,88.843700


4.) Read human gene2refseq database and extract gene identifiers

In [4]:
needed_refseq = set(
    mirdb_high_conf["RefSeq"]
    .dropna()
    .astype(str)
    .str.replace(r"\.\d+$", "", regex=True)
)

print("RefSeq accessions needed:", len(needed_refseq))

# Read the smaller human RefSeq annotation file
chunk = pd.read_csv(
    "data/GCF_000001405.40_GRCh38.p14_feature_table.txt.gz",
    sep="\t",
    compression="gzip",
    dtype=str,
    usecols=[
        "product_accession",
        "GeneID",
        "symbol"
    ]
)

# Remove accession version, e.g. NM_001130845.2 -> NM_001130845
chunk["RefSeq"] = (
    chunk["product_accession"]
    .str.replace(r"\.\d+$", "", regex=True)
)

# Retain only accessions occurring in our miRDB network
chunk = chunk[
    chunk["RefSeq"].isin(needed_refseq)
].copy()

# Rename column to match our original notebook
chunk = chunk.rename(columns={"symbol": "Symbol"})

# Keep the same output structure as before
mapping_chunks = []

if len(chunk) > 0:
    mapping_chunks.append(
        chunk[["RefSeq", "GeneID", "Symbol"]]
    )

print("Matching annotation records:", len(chunk))
print("Matched RefSeq accessions:", chunk["RefSeq"].nunique())
print("Unmatched RefSeq accessions:",
      len(needed_refseq - set(chunk["RefSeq"])))

RefSeq accessions needed: 38380
Matching annotation records: 42764
Matched RefSeq accessions: 37928
Unmatched RefSeq accessions: 452


Combine the retained records:

In [5]:
refseq_gene_map = pd.concat(
    mapping_chunks,
    ignore_index=True
).drop_duplicates()

refseq_gene_map = refseq_gene_map.rename(
    columns={"Symbol": "Gene"}
)

print("Relevant mapping records:", len(refseq_gene_map))
print("Mapped RefSeq accessions:",
      refseq_gene_map["RefSeq"].nunique())
print("Unique mapped genes:",
      refseq_gene_map["Gene"].nunique())

refseq_gene_map.head()

Relevant mapping records: 37928
Mapped RefSeq accessions: 37928
Unique mapped genes: 16527


,RefSeq,GeneID,Gene
0,NM_152486,148398,SAMD11
1,NM_015658,26155,NOC2L
2,NM_198317,339451,KLHL17
3,NM_001160184,84069,PLEKHN1
4,NM_032129,84069,PLEKHN1


Check for ambiguous mappings:

In [6]:
ambiguous_refseq = (
    refseq_gene_map.groupby("RefSeq")["Gene"]
    .nunique()
    .loc[lambda x: x > 1]
)

print("RefSeq accessions mapping to multiple genes:",
      len(ambiguous_refseq))

ambiguous_refseq.head()

RefSeq accessions mapping to multiple genes: 0


Series([], Name: Gene, dtype: int64)

5.) Map to gene identifiers and a column and save in file "mirdb_refseq_to_gene_symbol.csv"

In [7]:
mirdb_with_genes = mirdb_high_conf.merge(
    refseq_gene_map[["RefSeq", "Gene"]],
    on="RefSeq",
    how="left"
)

print("High-confidence predictions:", len(mirdb_with_genes))
print("Predictions mapped to genes:",
      mirdb_with_genes["Gene"].notna().sum())
print("Unmapped predictions:",
      mirdb_with_genes["Gene"].isna().sum())
print("Unique mapped genes:",
      mirdb_with_genes["Gene"].nunique())

#save to file 
refseq_gene_map.to_csv(
    "data/mirdb_refseq_to_gene_symbol.csv",
    index=False
)

High-confidence predictions: 812240
Predictions mapped to genes: 807022
Unmapped predictions: 5218
Unique mapped genes: 16527


6.) Check how many target genes occur in our PFC expression data. We look at the annotations which we have to load first. 

In [8]:
# load pfc data

mirna_expr = pd.read_csv("data/pfc_mirna_expression.csv")
mrna_expr = pd.read_csv("data/pfc_mrna_gene_expression.csv")
mirna_annotation = pd.read_csv("data/pfc_mirna_annotation.csv")

# change wrong names
targetscan_to_mirdb = {
    "hsa-mir-101-3p.1": "hsa-mir-101-3p",
    "hsa-mir-124-3p.1": "hsa-mir-124-3p",
    "hsa-mir-126-3p.1": "hsa-mir-126-3p",
    "hsa-mir-140-3p.2": "hsa-mir-140-3p",
    "hsa-mir-142-3p.1": "hsa-mir-142-3p",
    "hsa-mir-383-5p.1": "hsa-mir-383-5p",
    "hsa-mir-411-5p.1": "hsa-mir-411-5p",
    "hsa-mir-873-5p.1": "hsa-mir-873-5p",
}

mirna_annotation["miRDB_name"] = (
    mirna_annotation["MiRBase ID"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(targetscan_to_mirdb)
)

#check changes
mirna_annotation.loc[
    mirna_annotation["miRDB_name"]
    != mirna_annotation["MiRBase ID"].astype("string").str.lower(),
    [
        "miRNA_original",
        "MiRBase ID",
        "miRDB_name",
        "ACCESSION",
        "Seed+m8"
    ]
]

,miRNA_original,MiRBase ID,miRDB_name,ACCESSION,Seed+m8
11,hsa-mir-101,hsa-miR-101-3p.1,hsa-mir-101-3p,MIMAT0000099,ACAGUAC
20,hsa-mir-124,hsa-miR-124-3p.1,hsa-mir-124-3p,MIMAT0000422,AAGGCAC
27,hsa-mir-126,hsa-miR-126-3p.1,hsa-mir-126-3p,MIMAT0000445,CGUACCG
44,hsa-mir-140-3p,hsa-miR-140-3p.2,hsa-mir-140-3p,MIMAT0004597,ACCACAG
46,hsa-mir-142-3p,hsa-miR-142-3p.1,hsa-mir-142-3p,MIMAT0000434,GUAGUGU
144,hsa-mir-383,hsa-miR-383-5p.1,hsa-mir-383-5p,MIMAT0000738,GAUCAGA
148,hsa-mir-411,hsa-miR-411-5p.1,hsa-mir-411-5p,MIMAT0003329,AGUAGAC
182,hsa-mir-873,hsa-miR-873-5p.1,hsa-mir-873-5p,MIMAT0004953,CAGGAAC


Restrict annotation to measured miRNAs

In [9]:
measured_mirnas = set(
    mirna_expr["ID_REF"]
    .dropna()
    .astype(str)
    .str.strip()
)


8.) Map to seeds

In [10]:

mirna_seed_map = mirna_annotation[
    mirna_annotation["miRNA_original"]
    .astype(str)
    .str.strip()
    .isin(measured_mirnas)
][
    ["miRDB_name", "Seed+m8"]
].dropna().copy()

mirna_seed_map["miRNA_key"] = (
    mirna_seed_map["miRDB_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mirna_seed_map = (
    mirna_seed_map[
        ["miRNA_key", "Seed+m8"]
    ]
    .rename(columns={"Seed+m8": "Seed"})
    .drop_duplicates()
)

print("Measured miRNA names:",
      mirna_seed_map["miRNA_key"].nunique())
print("Measured seed families:",
      mirna_seed_map["Seed"].nunique())

mirna_seed_map.head()

Measured miRNA names: 197
Measured seed families: 157


,miRNA_key,Seed
0,hsa-let-7a-5p,GAGGUAG
1,hsa-let-7b-5p,GAGGUAG
2,hsa-let-7b-3p,UAUACAA
3,hsa-let-7c-5p,GAGGUAG
4,hsa-let-7d-5p,GAGGUAG


9.) Attach seed families to mirdb

In [11]:
mirdb_with_genes["miRNA_key"] = (
    mirdb_with_genes["miRNA"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mirdb_aligned = mirdb_with_genes.merge(
    mirna_seed_map,
    on="miRNA_key",
    how="inner",
    validate="many_to_one"
)

print("High-confidence predictions before miRNA matching:",
      len(mirdb_with_genes))

print("Predictions after measured-miRNA matching:",
      len(mirdb_aligned))

print("Measured miRNAs represented in miRDB:",
      mirdb_aligned["miRNA_key"].nunique())

print("Measured seed families represented in miRDB:",
      mirdb_aligned["Seed"].nunique())

print("Mapped target genes:",
      mirdb_aligned["Gene"].nunique())

mirdb_aligned[
    ["miRNA", "Seed", "RefSeq", "Gene", "miRDB_score"]
].head()

High-confidence predictions before miRNA matching: 812240
Predictions after measured-miRNA matching: 90081
Measured miRNAs represented in miRDB: 195
Measured seed families represented in miRDB: 156
Mapped target genes: 10318


,miRNA,Seed,RefSeq,Gene,miRDB_score
0,hsa-let-7a-5p,GAGGUAG,NM_003811,TNFSF9,86.271200
1,hsa-let-7a-5p,GAGGUAG,NM_001879,MASP1,83.991512
2,hsa-let-7a-5p,GAGGUAG,NM_001278205,NaN,90.325432
3,hsa-let-7a-5p,GAGGUAG,NM_173686,C8orf58,95.064780
4,hsa-let-7a-5p,GAGGUAG,NM_001204416,RGS6,87.908000


measured miRNA not found in miRDB

In [12]:
missing_mirnas = sorted(
    set(mirna_seed_map["miRNA_key"])
    - set(mirdb_aligned["miRNA_key"])
)

print("Unmatched miRNAs:", len(missing_mirnas))
missing_mirnas

Unmatched miRNAs: 2


['hsa-mir-137', 'hsa-mir-320a']

12.) Intersect with measured PFC genes and export the analysis-ready miRDB network

In [13]:
measured_genes = set(
    mrna_expr["Gene"]
    .dropna()
    .astype(str)
    .str.strip()
)

mirdb_measured = mirdb_aligned[
    mirdb_aligned["Gene"].isin(measured_genes)
].copy()

mirdb_network = (
    mirdb_measured
    .groupby(["Seed", "Gene"], as_index=False)
    .agg(
        miRDB_score=("miRDB_score", "max"),
        matched_miRNAs=(
            "miRNA_key",
            lambda values: ";".join(sorted(set(values)))
        ),
        n_matched_miRNAs=("miRNA_key", "nunique")
    )
)

print("Final miRDB edges:", len(mirdb_network))
print("Seed families:", mirdb_network["Seed"].nunique())
print("Target genes:", mirdb_network["Gene"].nunique())
print("Measured miRNAs retained:",
      mirdb_measured["miRNA_key"].nunique())
print("Duplicate edges:",
      mirdb_network.duplicated(["Seed", "Gene"]).sum())

mirdb_network.to_csv(
    "data/pfc_network_mirdb_high_confidence.csv",
    index=False
)

mirdb_network.head()

Final miRDB edges: 24111
Seed families: 156
Target genes: 7039
Measured miRNAs retained: 195
Duplicate edges: 0


,Seed,Gene,miRDB_score,matched_miRNAs,n_matched_miRNAs
0,AAAGCUG,ABCC5,86.944800,hsa-mir-320b;hsa-mir-320c,2
1,AAAGCUG,ABHD13,95.850965,hsa-mir-320b;hsa-mir-320c,2
2,AAAGCUG,ACBD3,84.855284,hsa-mir-320b;hsa-mir-320c,2
3,AAAGCUG,ACOT7,83.064000,hsa-mir-320b;hsa-mir-320c,2
4,AAAGCUG,ACTL6A,89.944959,hsa-mir-320b;hsa-mir-320c,2
